In [ ]:
from fipy import CellVariable, TransientTerm, DiffusionTerm, ExponentialConvectionTerm, Gmsh3D, VTKViewer
from fipy.solvers.scipy import LinearPCGSolver, LinearGMRESSolver
from fipy.tools import numerix
import matplotlib.pyplot as plt
import sys
from mpl_toolkits.mplot3d import Axes3D

In [3]:
# PHYSICAL CONSTANTS
kBT = 4.1E-21           # Thermal energy at room temp in J
e_charge = 1.6E-19      # elementary charge in C
epsilon_0 = 8.85E-12    # permitivity of free space in C/VM
N_A = 6.22E23           # Avagadro's Number

""" 
Most physical parameters are locked in place during mesh generation, while others aren't.
The following are the only parameters we are free to vary at this stage
"""
Delta_V = 0

In [4]:
def read_metadata(filepath):
    """Parse the simulation metadata file into a dict of {key: float}."""
    metadata = {}
    with open(filepath) as f:
        for line in f:
            line = line.strip()          # remove trailing newline / whitespace
            if not line or line.startswith("#"):
                continue                  # skip blank lines and comments
            key, value = line.split()     # splits on whitespace into exactly 2 pieces
            metadata[key] = float(value)
    return metadata

In [ ]:
# --- Domain -----------------------------------------------------------
mesh = Gmsh3D("final_meshes/dumbell.msh") # mesh file name here
parameters = read_metadata("final_meshes/dumbell_params.dat") # parameter file name here

print(mesh.physicalFaces)

# --- unit conversions
"""
r0 = 10nm -> nanopore length
c0 = 10 mol/m^3 = 6.022 r0^-3 -> bulk concentration density
V0 = kBT/e = 0.0257 -> thermal voltage
D0 = 2E-9 m^2/s -> diffusivity of K+ ions in water
t0 = r0^2/D0 = 5E-8 s -> characteristic diffusion time
"""

r0 = parameters["unit_length"]
c0 = parameters["unit_concentration"]
V0 = parameters["unit_voltage"]
D0 = parameters["unit_diffusivity"]
epsilon0 = parameters["unit_permitivity"]
t0 = parameters["unit_time"]
e0 = parameters["unit_charge"]

# --- Parameters (nondimensionalized) --------------------------
D_pos = parameters["diffusivity"] / D0
D_neg = parameters["diffusivity"] / D0                    # diffusivities
z_pos, z_neg = 1, -1                                      # valences
epsilon = parameters["relative_permitivity"] * epsilon_0 / epsilon0
sigma = parameters["surface_charge"] * (r0**2 / e0)       # surface charge density
Delta_V /= V0                                             # normalize the potential difference

# --- Variables ----------------------------------------------------------
potential = CellVariable(name="potential", mesh=mesh, value=0.0)
c_pos = CellVariable(name="cation concentration", mesh=mesh, value=1.0, hasOld=True)
c_neg = CellVariable(name="anion concentration", mesh=mesh, value=1.0, hasOld=True)

{'electrode_bottom': (FaceVariable(value=array([0, 0, 0, ..., 0, 0, 0], shape=(2966811,), dtype=int32), mesh=Gmsh3D()) == 2), 'electrode_top': (FaceVariable(value=array([0, 0, 0, ..., 0, 0, 0], shape=(2966811,), dtype=int32), mesh=Gmsh3D()) == 3), 'membrane_bottom': (FaceVariable(value=array([0, 0, 0, ..., 0, 0, 0], shape=(2966811,), dtype=int32), mesh=Gmsh3D()) == 4), 'membrane_top': (FaceVariable(value=array([0, 0, 0, ..., 0, 0, 0], shape=(2966811,), dtype=int32), mesh=Gmsh3D()) == 5), 'pore_wall': (FaceVariable(value=array([0, 0, 0, ..., 0, 0, 0], shape=(2966811,), dtype=int32), mesh=Gmsh3D()) == 6), 'reservoir_wall_bottom': (FaceVariable(value=array([0, 0, 0, ..., 0, 0, 0], shape=(2966811,), dtype=int32), mesh=Gmsh3D()) == 7), 'reservoir_wall_top': (FaceVariable(value=array([0, 0, 0, ..., 0, 0, 0], shape=(2966811,), dtype=int32), mesh=Gmsh3D()) == 8)}


In [6]:
# BOUNDARY CONDITIONS

# potential applied across pore
potential.constrain(Delta_V, mesh.physicalFaces["electrode_top"])
potential.constrain(0.0, mesh.physicalFaces["electrode_bottom"])

#concentrations converge to the bulk value far away from the pore
c_pos.constrain(1.0, mesh.physicalFaces["electrode_top"])
c_pos.constrain(1.0, mesh.physicalFaces["electrode_bottom"])
c_neg.constrain(1.0, mesh.physicalFaces["electrode_top"])
c_neg.constrain(1.0, mesh.physicalFaces["electrode_bottom"])

# surface charge
potential.faceGrad.constrain(-(sigma/epsilon) * mesh.faceNormals, where=mesh.physicalFaces["membrane_top"])
potential.faceGrad.constrain(-(sigma/epsilon) * mesh.faceNormals, where=mesh.physicalFaces["membrane_bottom"])
potential.faceGrad.constrain(-(sigma/epsilon) * mesh.faceNormals, where=mesh.physicalFaces["pore_wall"])

In [7]:
# --- Equations ------------------------------------------------------------
# Poisson: elliptic, no TransientTerm. Charge density depends on the
# *current* concentration guesses -- this is the nonlinear coupling.
charge_density = z_pos * c_pos + z_neg * c_neg
poisson_eq = DiffusionTerm(coeff=epsilon) == -charge_density

# Nernst-Planck: transient diffusion-convection. The "velocity" driving each
# species is built from the potential gradient -- this is where
# ExponentialConvectionTerm gives you the exponential-fitting (Slotboom-like)
# robustness discussed earlier, without needing a manual variable transform.
pos_eq = TransientTerm() == DiffusionTerm(coeff=D_pos) \
    + ExponentialConvectionTerm(coeff=z_pos * D_pos * potential.faceGrad)
neg_eq = TransientTerm() == DiffusionTerm(coeff=D_neg) \
    + ExponentialConvectionTerm(coeff=z_neg * D_neg * potential.faceGrad)

In [ ]:
# --- Time stepping with an inner Gummel loop -------------------------------
dt = 1e-2
n_timesteps = 200
max_gummel_iters = 30
gummel_tol = 1e-6

# Iterative solvers: the default direct LU solver (SuperLU) runs out of room on
# large 3D meshes. Poisson is symmetric -> PCG; Nernst-Planck is not (the
# convection term breaks symmetry) -> GMRES.
poisson_solver = LinearPCGSolver(tolerance=1e-10, iterations=5000)
np_solver = LinearGMRESSolver(tolerance=1e-10, iterations=5000)

for step in range(n_timesteps):

    for gummel_iter in range(max_gummel_iters):
        potential_before = potential.value.copy()

        # 1. Solve Poisson using the current concentration guesses.
        #    No TransientTerm involved -- this is just a linear elliptic solve
        #    once concentrations are frozen for this iteration.
        poisson_eq.solve(var=potential, solver=poisson_solver)

        # 2. Sweep (not solve!) the Nernst-Planck equations using the updated
        #    potential. sweep() re-linearizes and re-solves *without*
        #    advancing time, which is exactly what Gummel iteration needs.
        pos_residual = pos_eq.sweep(var=c_pos, dt=dt, solver=np_solver)
        neg_residual = neg_eq.sweep(var=c_neg, dt=dt, solver=np_solver)

        # 3. Check self-consistency: has the potential (and, implicitly via
        #    the residuals, the concentrations) stopped changing?
        potential_change = numerix.abs(potential.value - potential_before).max()
        if potential_change < gummel_tol:
            break
    else:
        print(f"step {step}: Gummel loop did not converge in "
              f"{max_gummel_iters} iterations (last change={potential_change:.2e})")

    # Only now, after the inner loop is self-consistent, advance in time.
    c_pos.updateOld()
    c_neg.updateOld()

    print(str(step) + "/" + str(n_timesteps))

print("Done. potential range:", potential.value.min(), potential.value.max())
print("cation range:", c_pos.value.min(), c_pos.value.max())
print("anion range:", c_neg.value.min(), c_neg.value.max())

In [ ]:
vals = potential.value
coords = mesh.cellCenters  # shape (3, N)

x, y, z = coords[0], coords[1], coords[2]

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

p = ax.scatter(x, y, z, c=vals, cmap='viridis', s=3)

fig.colorbar(p, ax=ax, shrink=0.6)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

plt.title("Potential")
plt.show()


In [ ]:
vals = c_neg.value
coords = mesh.cellCenters  # shape (3, N)

x, y, z = coords[0], coords[1], coords[2]

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

p = ax.scatter(x, y, z, c=vals, cmap='viridis', s=3)

fig.colorbar(p, ax=ax, shrink=0.6)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

plt.title("K+ ion concentration")
plt.show()


In [ ]:
vals = c_pos.value
coords = mesh.cellCenters  # shape (3, N)

x, y, z = coords[0], coords[1], coords[2]

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

p = ax.scatter(x, y, z, c=vals, cmap='viridis', s=3)

fig.colorbar(p, ax=ax, shrink=0.6)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

plt.title("Cl- Ion Concentration")
plt.show()

In [ ]:
viewer = VTKViewer(vars=(c_pos, c_neg, potential))
viewer.plot(filename="exmplePoreSolution.vtk")